[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/07_logistic_regression/03_regularized_logistic_regression.ipynb)

# Bài 03 — Regularized Logistic Regression và Feature Mapping

## 1. Overfitting với bài toán phân loại

Tương tự Linear Regression, Logistic Regression có thể underfit (ranh giới quá đơn giản) hoặc overfit (ranh giới uốn éo bám theo từng điểm, kể cả nhiễu):

<img src="images/overfitting_2.png" style="width:65%;height:65%;">

Ranh giới của Logistic Regression với feature tuyến tính luôn là đường thẳng. Để có ranh giới **cong**, ta **ánh xạ feature** (feature mapping) sang không gian đa thức — và khi đó cần **regularization** để không overfit.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/07_logistic_regression

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from ml_course.datasets import make_classification_2d
from ml_course.visualization import plot_decision_boundary, set_style

set_style()

## 2. Cost function có regularization

$$J(\theta) = -\frac{1}{m}\sum_{i=1}^{m}\Big[y^{(i)}\log h_\theta(x^{(i)}) + (1 - y^{(i)})\log\big(1 - h_\theta(x^{(i)})\big)\Big] + \frac{\lambda}{2m}\sum_{j=1}^{n}\theta_j^2$$

**Không phạt $\theta_0$.** Hệ số $\frac{\lambda}{2m}$ có số 2 ở mẫu để triệt tiêu khi lấy đạo hàm.

**Dạng vector:** $J(\theta) = -\frac1m\left[y^\top\log h + (1 - y)^\top\log(1 - h)\right] + \frac{\lambda}{2m}\sum_{j=1}^n\theta_j^2$

## 3. Gradient Descent (giống Linear Regression)

$$\theta_0 := \theta_0 - \alpha\frac{1}{m}\sum_{i=1}^m\left(h_\theta(x^{(i)}) - y^{(i)}\right)x_0^{(i)}$$

$$\theta_j := \theta_j - \alpha\left[\frac{1}{m}\sum_{i=1}^m\left(h_\theta(x^{(i)}) - y^{(i)}\right)x_j^{(i)} + \frac{\lambda}{m}\theta_j\right], \quad j = 1, \dots, n$$

**Dạng vector:** $\theta := \theta - \frac{\alpha}{m}\left[X^\top\left(g(X\theta) - \vec y\right) + \lambda\theta'\right]$, với $\theta'$ là $\theta$ có phần tử đầu bằng 0.

## 4. Feature mapping + λ: thực nghiệm

Dữ liệu dạng hai vòng tròn lồng nhau — không thể tách bằng đường thẳng. Ánh xạ $(x_1, x_2)$ sang mọi đơn thức bậc ≤ 6 (giống bài tập vi mạch của khoá AIVN/Andrew Ng — 28 feature), rồi thay đổi $\lambda$:

In [ ]:
X, y = make_classification_2d("circles", n=200, noise=0.35, seed=0)
X_val, y_val = make_classification_2d("circles", n=2000, noise=0.35, seed=1)

fig, axes = plt.subplots(1, 4, figsize=(22, 4.6))
configs = [("Tuyến tính (không mapping)", 1, 1.0), ("Bậc 6, λ ≈ 0", 6, 1e-6), ("Bậc 6, λ = 1", 6, 1.0), ("Bậc 6, λ = 100", 6, 100.0)]
for ax, (title, degree, lam) in zip(axes, configs):
    model = make_pipeline(PolynomialFeatures(degree), StandardScaler(), LogisticRegression(C=1 / lam, max_iter=20000))
    model.fit(X, y)
    plot_decision_boundary(model, X, y, ax=ax)
    ax.set_title(f"{title}\ntrain = {model.score(X, y):.2f} | validation = {model.score(X_val, y_val):.2f}", fontsize=10)
plt.show()

| λ | Hiện tượng |
|---|---|
| Tuyến tính | **underfit** — đường thẳng không thể bao quanh vòng tròn |
| Bậc 6, λ ≈ 0 | **overfit** — ranh giới uốn éo, accuracy train cao nhưng validation thấp hơn |
| Bậc 6, λ = 1 | vừa phải — ranh giới gần tròn, validation tốt nhất |
| Bậc 6, λ = 100 | **underfit** — phạt quá mạnh, ranh giới gần như vô nghĩa |

Trong scikit-learn: `LogisticRegression(C=1/λ)`.

## 5. Chọn λ (hay C) bằng cross-validation

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

cv_model = make_pipeline(PolynomialFeatures(6), StandardScaler(),
                         LogisticRegressionCV(Cs=np.logspace(-3, 3, 13), cv=5, max_iter=20000)).fit(X, y)
best_C = cv_model[-1].C_[0]
print(f"C tốt nhất = {best_C:.3g} (λ ≈ {1 / best_C:.3g}) | accuracy validation = {cv_model.score(X_val, y_val):.3f}")

## 6. L1 regularization cho Logistic Regression *(bổ sung)*

Như Lasso, phạt L1 (`penalty="l1"`, solver `liblinear` hoặc `saga`) đưa nhiều hệ số về đúng 0 → chọn feature tự động. Với 28 feature đa thức, xem bao nhiêu feature được giữ lại:

In [ ]:
for C in [0.05, 0.5, 5.0]:
    l1 = make_pipeline(PolynomialFeatures(6), StandardScaler(),
                       LogisticRegression(penalty="l1", C=C, solver="liblinear", max_iter=5000)).fit(X, y)
    nz = np.sum(np.abs(l1[-1].coef_) > 1e-8)
    print(f"C = {C:<5} → {nz:2d}/28 hệ số khác 0 | accuracy validation = {l1.score(X_val, y_val):.3f}")

L1 tìm ra rằng chỉ cần vài feature (chủ yếu $x_1^2, x_2^2$ — đúng phương trình vòng tròn!) để phân loại tốt.

## 🧠 Kiểm tra nhanh

1. Vì sao feature mapping đa thức cho phép Logistic Regression có ranh giới cong?
2. Feature mapping bậc 6 với 2 feature tạo bao nhiêu feature (kể cả hằng số)?
3. `C` tăng thì decision boundary trở nên phức tạp hơn hay đơn giản hơn?

## Tài liệu tham khảo

[1] [Overfitting — Machine Learning Cơ Bản](https://machinelearningcoban.com/2017/03/04/overfitting/) · [2] [CS229 — Machine Learning](https://www.coursera.org/learn/machine-learning/resources/Zi29t)

---
<sub>Bài gốc: *Regularization* (phần Regularized Logistic Regression) — [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 4 (MIT License). Đã đính chính hệ số λ/m trong gradient; bổ sung: thực nghiệm feature mapping × λ, chọn C bằng CV, L1 regularization.</sub>